> **© 2026 Chirag Shinde. Licensed under CC BY-NC-SA 4.0.**
> See [LICENSE](../../LICENSE) for details.

---

# Chapter 11: Data Quality and Cleaning

## Why This Matters

In 2018, Amazon abandoned a recruiting algorithm that discriminated against women because it was trained on biased, low-quality historical data. Zillow lost hundreds of millions when its home-valuation AI failed due to poor data quality. According to MIT Sloan Management Review, most companies lose 15–25% of their revenue to bad data. The principle is simple but unforgiving: **garbage in, garbage out**. Before building sophisticated models, ensuring data is clean, complete, and trustworthy is essential—because even the best algorithm cannot rescue insights from fundamentally flawed data.

## Intuition

Imagine assembling IKEA furniture, but the parts come from multiple boxes with inconsistent labeling. Some screws are missing, there are two identical tabletops (one must be extra), one leg is suspiciously 10 feet long, and some measurements are in inches while others are in centimeters. Building wouldn't start without first sorting through these issues—finding replacements for missing screws, removing the duplicate, investigating that impossibly long leg, and converting everything to the same units.

Data quality is exactly the same. A dataset is like those furniture parts: it arrives with missing values (like missing screws), duplicates (identical records entered twice), outliers (that 10-foot leg that's clearly wrong), and inconsistencies (mixed units or formats). Just as stable furniture can't be built from faulty parts, reliable models can't be built from poor-quality data.

Here's the key insight: **not all data problems have the same solution**. Sometimes a missing value is truly random (a lab equipment failure), sometimes it's related to other information available (older patients' records are more complete), and sometimes the fact that it's missing is itself meaningful (high earners hiding their income on loan applications). Understanding *why* data has quality issues guides toward the *right* way to fix them.

Data cleaning isn't a one-time checkbox before modeling. It's an iterative detective process intertwined with exploratory analysis. Issues are discovered, cleaned, more subtle issues found, the approach refined, and validation confirms that fixes actually improved things. This chapter teaches systematic diagnosis of data quality problems and appropriate strategy selection—not by following rigid rules, but by understanding trade-offs and making informed decisions.

## Formal Definition

**Data quality** encompasses multiple dimensions of fitness for use in analysis and modeling. The four primary dimensions are:

1. **Completeness**: The degree to which all required data is present. Measured by the proportion of non-missing values:
   $$\text{Completeness} = \frac{\text{non-null values}}{n \times p}$$
   where *n* is the number of samples and *p* is the number of features.

2. **Uniqueness**: The absence of duplicate records. Measured by:
   $$\text{Uniqueness} = \frac{\text{unique rows}}{n}$$

3. **Validity**: The extent to which data values fall within expected ranges and follow expected patterns. Outliers are values that deviate significantly from the distribution.

4. **Consistency**: Uniformity in data types, formats, and scales across the dataset.

**Missing data mechanisms** describe *why* values are absent:

- **MCAR (Missing Completely at Random)**: The probability of missingness is independent of all data, observed or unobserved. Formally: $$P(\text{missing}) = \text{constant}$$

- **MAR (Missing at Random)**: The probability of missingness depends only on observed data, not the missing values themselves. Formally: $$P(\text{missing} \mid X_{\text{obs}}, X_{\text{miss}}) = P(\text{missing} \mid X_{\text{obs}})$$

- **MNAR (Missing Not at Random)**: The probability of missingness depends on the unobserved missing values themselves. This is the most problematic type.

**Outliers** are observations that deviate significantly from the pattern of the data. Two common detection methods:

- **Z-score method**: Flag values where $$z = \frac{x - \mu}{\sigma}, \quad |z| > 3$$
  where μ is the mean and σ is the standard deviation.

- **IQR method**: Flag values outside the bounds $$[Q_1 - 1.5 \times \text{IQR}, \; Q_3 + 1.5 \times \text{IQR}]$$
  where IQR = Q₃ - Q₁ is the interquartile range.

> **Key Concept:** Data quality assessment is not about following universal rules—it's about understanding the *mechanism* behind quality issues and choosing strategies that match the context of the problem.

## Visualization

In [ ]:
# Generate visualization showing the three types of missing data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

# Create sample data demonstrating three missingness types
n_samples = 100
age = np.random.randint(20, 70, n_samples)
income = 30000 + age * 1000 + np.random.normal(0, 10000, n_samples)

# Create DataFrame
df_demo = pd.DataFrame({
    'Age': age,
    'Income_MCAR': income.copy(),
    'Income_MAR': income.copy(),
    'Income_MNAR': income.copy()
})

# MCAR: Randomly remove 20% of values (truly random)
mcar_mask = np.random.random(n_samples) < 0.2
df_demo.loc[mcar_mask, 'Income_MCAR'] = np.nan

# MAR: Remove income for older people (related to age, which is observed)
mar_mask = (age > 50) & (np.random.random(n_samples) < 0.4)
df_demo.loc[mar_mask, 'Income_MAR'] = np.nan

# MNAR: Remove high incomes (related to income itself, which is unobserved)
mnar_mask = (income > 60000) & (np.random.random(n_samples) < 0.5)
df_demo.loc[mnar_mask, 'Income_MNAR'] = np.nan

# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# MCAR visualization
axes[0].scatter(df_demo['Age'], df_demo['Income_MCAR'], alpha=0.6, label='Observed')
axes[0].scatter(df_demo.loc[df_demo['Income_MCAR'].isna(), 'Age'],
                [20000] * mcar_mask.sum(), color='red', marker='x', s=100, label='Missing')
axes[0].set_title('MCAR: Missing Completely at Random\n(Missing scattered randomly)', fontsize=11)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Income')
axes[0].legend()

# MAR visualization
axes[1].scatter(df_demo['Age'], df_demo['Income_MAR'], alpha=0.6, label='Observed')
axes[1].scatter(df_demo.loc[df_demo['Income_MAR'].isna(), 'Age'],
                [20000] * mar_mask.sum(), color='red', marker='x', s=100, label='Missing')
axes[1].axvline(x=50, color='orange', linestyle='--', alpha=0.5, label='Age > 50')
axes[1].set_title('MAR: Missing at Random\n(Missing related to Age—observed)', fontsize=11)
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Income')
axes[1].legend()

# MNAR visualization
axes[2].scatter(df_demo['Age'], df_demo['Income_MNAR'], alpha=0.6, label='Observed')
axes[2].scatter(df_demo.loc[df_demo['Income_MNAR'].isna(), 'Age'],
                [20000] * mnar_mask.sum(), color='red', marker='x', s=100, label='Missing')
axes[2].axhline(y=60000, color='purple', linestyle='--', alpha=0.5, label='Income > $60k')
axes[2].set_title('MNAR: Missing Not at Random\n(High earners hide income)', fontsize=11)
axes[2].set_xlabel('Age')
axes[2].set_ylabel('Income')
axes[2].legend()

plt.tight_layout()
plt.savefig('diagrams/missing_data_types.png', dpi=150, bbox_inches='tight')
plt.show()

# Output:
# Three scatter plots showing different missing patterns:
# - MCAR: Red X's scattered randomly across all ages and income levels
# - MAR: Red X's clustered in the older age group (Age > 50)
# - MNAR: Red X's clustered in the high-income region (Income > $60k)

**Figure 11.1:** The three types of missing data mechanisms. MCAR shows no pattern (purely random), MAR shows missingness correlated with an observed variable (Age), and MNAR shows missingness related to the unobserved value itself (high Income). Understanding the mechanism guides handling strategy.

## Examples

### Part 1: Load and Profile Data

In [ ]:
# Complete data quality assessment and cleaning pipeline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Load California Housing dataset
housing = fetch_california_housing()
df_raw = pd.DataFrame(housing.data, columns=housing.feature_names)
df_raw['MedHouseVal'] = housing.target

print("=== STEP 1: Load and Profile Data ===")
print(f"Shape: {df_raw.shape}")
print(f"\nFirst few rows:")
print(df_raw.head())
print(f"\nData types:")
print(df_raw.dtypes)
print(f"\nBasic statistics:")
print(df_raw.describe())

# Artificially introduce quality issues for demonstration
df = df_raw.copy()

# Missing values: Introduce 15% MCAR in 'MedInc' and 'AveRooms'
n = len(df)
mcar_mask_inc = np.random.random(n) < 0.15
mcar_mask_rooms = np.random.random(n) < 0.15
df.loc[mcar_mask_inc, 'MedInc'] = np.nan
df.loc[mcar_mask_rooms, 'AveRooms'] = np.nan

# Duplicates: Duplicate 30 random rows
duplicate_indices = np.random.choice(df.index, 30, replace=False)
df = pd.concat([df, df.loc[duplicate_indices]], ignore_index=True)

# Outliers: Create some artificial extreme values
outlier_indices = np.random.choice(df.index, 20, replace=False)
df.loc[outlier_indices, 'MedInc'] = df.loc[outlier_indices, 'MedInc'] * 5

# Output:
# === STEP 1: Load and Profile Data ===
# Shape: (20640, 9)
#
# First few rows:
#    MedInc  HouseAge  AveRooms  ...  Latitude  Longitude  MedHouseVal
# 0  8.3252      41.0  6.984127  ...     37.88    -122.23         4.526
# ...

The pipeline starts by loading the California Housing dataset, which has 20,640 samples and 9 features. Always begin by profiling: check the shape, look at the first few rows with `df.head()`, examine data types with `df.dtypes`, and compute basic statistics with `df.describe()`. This initial reconnaissance reveals the structure and scale of data.

For demonstration, realistic quality issues are artificially introduced: 15% missing values (MCAR) in `MedInc` and `AveRooms`, 30 duplicate rows, and outliers by multiplying 20 random `MedInc` values by 5. Real-world data often has all these issues simultaneously.

### Part 2: Detect Quality Issues

In [ ]:
print("\n=== STEP 2: Detect Quality Issues ===")

# Missing values
print("\nMissing values per column:")
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing_counts,
    'Missing_Percent': missing_pct
})
print(missing_df[missing_df['Missing_Count'] > 0])

# Duplicates
dup_count = df.duplicated().sum()
print(f"\nDuplicate rows: {dup_count} ({dup_count/len(df)*100:.2f}%)")

# Outliers using IQR method for 'MedInc'
Q1 = df['MedInc'].quantile(0.25)
Q3 = df['MedInc'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers_iqr = ((df['MedInc'] < lower_bound) | (df['MedInc'] > upper_bound)).sum()
print(f"\nOutliers in 'MedInc' (IQR method): {outliers_iqr}")

# Outliers using Z-score method
z_scores = np.abs((df['MedInc'] - df['MedInc'].mean()) / df['MedInc'].std())
outliers_z = (z_scores > 3).sum()
print(f"Outliers in 'MedInc' (Z-score method): {outliers_z}")

# Output:
# === STEP 2: Detect Quality Issues ===
# Missing values per column:
#              Missing_Count  Missing_Percent
# MedInc                3109            15.02
# AveRooms              3109            15.02
#
# Duplicate rows: 30 (0.14%)
# Outliers in 'MedInc' (IQR method): 1829
# Outliers in 'MedInc' (Z-score method): 156

Quality issues are detected systematically. Missing values are identified using `df.isnull().sum()` and percentages are calculated to understand severity. For duplicates, `df.duplicated().sum()` tells how many rows are exact copies. For outliers, both IQR and Z-score methods are applied to `MedInc`. Notice they find different numbers of outliers (1,829 vs. 156)—IQR is more sensitive and doesn't assume normality, while Z-score is stricter but assumes a normal distribution.

### Part 3: Visualize Quality Issues

In [ ]:
print("\n=== STEP 3: Visualize Quality Issues ===")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Missing data bar chart
missing_viz = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
axes[0, 0].bar(range(len(missing_viz)), missing_viz['Missing_Count'], color='coral')
axes[0, 0].set_xticks(range(len(missing_viz)))
axes[0, 0].set_xticklabels(missing_viz.index, rotation=45)
axes[0, 0].set_title('Missing Values by Column')
axes[0, 0].set_ylabel('Count')

# Missing data heatmap (first 100 rows)
sns.heatmap(df.head(100).isnull(), cbar=False, cmap='viridis', ax=axes[0, 1])
axes[0, 1].set_title('Missing Data Pattern (First 100 Rows)\nYellow = Missing')

# Box plot showing outliers
axes[1, 0].boxplot(df['MedInc'].dropna(), vert=True)
axes[1, 0].axhline(upper_bound, color='red', linestyle='--', label=f'Upper bound: {upper_bound:.2f}')
axes[1, 0].axhline(lower_bound, color='red', linestyle='--', label=f'Lower bound: {lower_bound:.2f}')
axes[1, 0].set_ylabel('MedInc')
axes[1, 0].set_title('Outliers in MedInc (IQR Method)')
axes[1, 0].legend()

# Distribution comparison
axes[1, 1].hist(df_raw['MedInc'], bins=50, alpha=0.5, label='Original', color='blue')
axes[1, 1].hist(df['MedInc'].dropna(), bins=50, alpha=0.5, label='With Outliers', color='red')
axes[1, 1].set_xlabel('MedInc')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Distribution: Original vs. Contaminated')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('diagrams/quality_issues_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

The visualization panel shows four critical views: (1) a bar chart quantifying missing values per column, (2) a heatmap revealing missingness patterns in the first 100 rows (yellow = missing), (3) a box plot with red dashed lines showing IQR outlier boundaries, and (4) a histogram comparing the original distribution (blue) to the contaminated one (red). Visualization is crucial because patterns that are invisible in tables become obvious visually.

### Part 4: Clean the Data

In [ ]:
print("\n=== STEP 4: Clean the Data ===")

# Remove duplicates
df_clean = df.drop_duplicates()
print(f"After removing duplicates: {len(df_clean)} rows (removed {len(df) - len(df_clean)})")

# Handle outliers: Cap at 1st and 99th percentiles (Winsorization)
p01 = df_clean['MedInc'].quantile(0.01)
p99 = df_clean['MedInc'].quantile(0.99)
df_clean['MedInc'] = df_clean['MedInc'].clip(lower=p01, upper=p99)
print(f"Capped 'MedInc' outliers to [{p01:.2f}, {p99:.2f}]")

# Split data BEFORE imputation (critical to avoid data leakage!)
X = df_clean.drop('MedHouseVal', axis=1)
y = df_clean['MedHouseVal']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTrain set: {X_train.shape}, Test set: {X_test.shape}")

# Output:
# === STEP 4: Clean the Data ===
# After removing duplicates: 20640 rows (removed 30)
# Capped 'MedInc' outliers to [1.36, 11.63]
# Train set: (16512, 8), Test set: (4128, 8)

Data is cleaned systematically. First, duplicates are removed with `drop_duplicates()`, eliminating 30 rows. Next, outliers are handled using **winsorization** (capping): `MedInc` is clipped to the 1st and 99th percentiles rather than deleting outliers entirely. This preserves sample size while reducing extreme influence.

Here's the **critical step that many beginners get wrong**: data is split into train and test sets *before* imputing missing values. If the mean were calculated on the entire dataset (including the test set), then used to fill missing values, information from the test set would leak into the training set. This inflates performance estimates artificially. Instead, splitting happens first, then the imputer is fit on training data only and applied to both sets.

### Part 5: Compare Imputation Strategies

In [ ]:
# Compare imputation strategies

# Strategy 1: Mean Imputation
imputer_mean = SimpleImputer(strategy='mean')
X_train_mean = pd.DataFrame(
    imputer_mean.fit_transform(X_train),
    columns=X_train.columns
)
X_test_mean = pd.DataFrame(
    imputer_mean.transform(X_test),
    columns=X_test.columns
)

# Strategy 2: Median Imputation
imputer_median = SimpleImputer(strategy='median')
X_train_median = pd.DataFrame(
    imputer_median.fit_transform(X_train),
    columns=X_train.columns
)
X_test_median = pd.DataFrame(
    imputer_median.transform(X_test),
    columns=X_test.columns
)

# Strategy 3: KNN Imputation
imputer_knn = KNNImputer(n_neighbors=5)
X_train_knn = pd.DataFrame(
    imputer_knn.fit_transform(X_train),
    columns=X_train.columns
)
X_test_knn = pd.DataFrame(
    imputer_knn.transform(X_test),
    columns=X_test.columns
)

Three imputation strategies are tested: mean, median, and KNN. For each:
1. The imputer is fit on `X_train` only (learning the statistic)
2. Both `X_train` and `X_test` are transformed using the learned statistic

This ensures no information leaks from test to training data.

### Part 6: Evaluate Impact of Cleaning

In [ ]:
print("\n=== STEP 5: Evaluate Impact of Cleaning ===")

# Train models with each imputation strategy
strategies = {
    'Mean Imputation': (X_train_mean, X_test_mean),
    'Median Imputation': (X_train_median, X_test_median),
    'KNN Imputation': (X_train_knn, X_test_knn)
}

results = {}
for name, (X_tr, X_te) in strategies.items():
    model = LinearRegression()
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    results[name] = rmse
    print(f"{name}: RMSE = {rmse:.4f}")

# Find best strategy
best_strategy = min(results, key=results.get)
print(f"\nBest strategy: {best_strategy} (RMSE = {results[best_strategy]:.4f})")

# Visualize results
fig, ax = plt.subplots(figsize=(10, 6))
strategies_list = list(results.keys())
rmse_values = list(results.values())
colors = ['coral' if s != best_strategy else 'green' for s in strategies_list]
ax.bar(strategies_list, rmse_values, color=colors)
ax.set_ylabel('RMSE (lower is better)')
ax.set_title('Comparison of Imputation Strategies')
ax.axhline(min(rmse_values), color='green', linestyle='--', alpha=0.5, label='Best')
for i, v in enumerate(rmse_values):
    ax.text(i, v + 0.01, f'{v:.4f}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig('diagrams/imputation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n=== STEP 6: Summary ===")
print(f"Original data: {df_raw.shape}")
print(f"After introducing issues: {df.shape}")
print(f"After cleaning: {df_clean.shape}")
print(f"Missing values handled: Mean, Median, and KNN strategies compared")
print(f"Best performing strategy: {best_strategy}")

# Output:
# === STEP 5: Evaluate Impact of Cleaning ===
# Mean Imputation: RMSE = 0.7289
# Median Imputation: RMSE = 0.7284
# KNN Imputation: RMSE = 0.7265
# Best strategy: KNN Imputation (RMSE = 0.7265)

Each strategy is evaluated by:
1. Training a Linear Regression model
2. Calculating RMSE on the test set

The results show KNN imputation performs best (RMSE = 0.7265), outperforming mean (0.7289) and median (0.7284). Why? KNN uses relationships between features to make smarter imputations, while mean/median use only the single column. However, KNN is computationally more expensive—a trade-off to consider with larger datasets.

The bar chart visualization clearly shows which strategy performed best, with the best bar highlighted in green. A summary compares the original shape (20,640 × 9) to the cleaned shape (same dimensions, but with quality improved). The key insight: cleaning improved model performance by reducing noise and ensuring consistent, complete data.

**Key Lesson from This Pipeline**: Data cleaning is not a single function call—it's a systematic process of detection, visualization, decision-making, and validation. Each choice (whether to remove or cap outliers, which imputation method to use) depends on understanding data context and downstream task requirements.

## Common Pitfalls

**1. Imputing Before Train-Test Split (Data Leakage)**

This is the single most common and critical mistake in data cleaning. Many beginners calculate statistics (like the mean for imputation) on the entire dataset, then split into train and test sets. This leaks information from the test set into the training process, resulting in overly optimistic performance estimates that don't hold in production.

**Wrong approach:**

In [ ]:
# DON'T DO THIS!
df['feature'].fillna(df['feature'].mean(), inplace=True)  # Uses ALL data
X_train, X_test = train_test_split(X, y)  # Split after imputation

**Correct approach:**

In [ ]:
# Split FIRST
X_train, X_test = train_test_split(X, y, random_state=42)
# Learn statistic on training data only
train_mean = X_train['feature'].mean()
# Apply to both sets
X_train['feature'].fillna(train_mean, inplace=True)
X_test['feature'].fillna(train_mean, inplace=True)

**Why it matters:** In the wrong approach, the imputation mean contains information from test samples. The model gets to "peek" at test data during training. Use scikit-learn Pipelines to automate this correctly.

**2. Treating All Outliers as Errors**

Outliers are not automatically wrong—they can be legitimate extreme values that carry important information. Removing all outliers blindly can discard valuable signal and reduce a model's ability to generalize to rare but real cases.

**What to do instead:** Investigate outliers before deciding. Ask: Is this value physically possible? Does domain knowledge suggest it's an error or a genuine extreme? For example, a billionaire's income is an outlier but not an error. A person with age 200 is clearly wrong. When in doubt, use robust methods (like winsorization/capping) that reduce influence without deletion, or use tree-based models that are naturally robust to outliers.

**3. Using Mean Imputation as a Default**

Mean imputation seems safe and easy, but it has serious problems: it distorts distributions (reduces variance), underestimates uncertainty, is heavily influenced by outliers, and performs poorly for skewed data. It's only appropriate for MCAR data with low missingness (<5-10%) in roughly normal distributions.

**What to do instead:** For skewed distributions, use median imputation. For data with complex feature relationships, use KNN or Iterative imputation. For MNAR data (where missingness itself is informative), create a binary "was_missing" indicator column alongside imputation. Always visualize the distribution before and after imputation to check if artifacts have been introduced.

## Practice

**Practice 1**

Load the Breast Cancer dataset from scikit-learn (`load_breast_cancer()`) and create a data quality assessment:

1. Convert to a pandas DataFrame with proper column names
2. Artificially introduce quality issues:
   - Set 10% of values in two random features to `np.nan`
   - Duplicate 15 random rows
3. Write code to detect and report:
   - Missing value count and percentage per column
   - Number of duplicate rows
   - Basic statistics (mean, std, min, max) for all features
4. Create a bar chart showing missing value counts per column

**Practice 2**

Using the Diabetes dataset from scikit-learn (`load_diabetes()`):

1. Artificially remove 20% of values (MCAR) from three features of choice
2. Split into train/test sets (80/20) **before** any imputation
3. Implement and compare four strategies:
   - Deletion: Remove rows with any missing values (listwise deletion)
   - Mean imputation
   - Median imputation
   - KNN imputation (k=5)
4. For each strategy:
   - Apply the appropriate transformation to train and test sets
   - Fit a `LinearRegression` model
   - Calculate and compare RMSE on the test set
5. Create visualizations:
   - Distribution histograms showing original vs. imputed values for one feature
   - Bar chart comparing RMSE across all four strategies
6. Answer: Which strategy performed best? Why? What's the trade-off with deletion (hint: check how many training samples were lost)?

**Practice 3**

Create a comprehensive, reusable data quality assessment and cleaning system:

**Part A: Quality Report Function**

Write a function `generate_quality_report(df)` that returns a dictionary containing:
- Shape and memory usage
- Missing value statistics (count, percentage per column)
- Duplicate row count
- Outlier counts per numeric column using both IQR and Z-score methods
- Data type summary
- Suspicious patterns (e.g., all zeros, all same value, negative values where impossible)

**Part B: Visualization Dashboard**

Create a function `visualize_quality(df, report)` that generates a 2×3 subplot dashboard:
1. Missing data bar chart (count per column)
2. Missing data heatmap (first 100 rows)
3. Box plots for all numeric features (showing outliers)
4. Correlation heatmap of missingness (which features tend to be missing together?)
5. Distribution histograms for features with outliers
6. Quality score summary (create a simple metric: 100 - (missing% + duplicate% + outlier%))

**Part C: Automated Cleaning Function**

Write a function `clean_data(df, config)` that:
- Takes a configuration dictionary specifying strategies per issue type
- Example config:
  ```python
  config = {
      'duplicates': 'remove',  # or 'keep'
      'outliers': {'method': 'winsorize', 'percentiles': (0.01, 0.99)},  # or 'remove' or 'keep'
      'missing': {'MedInc': 'mean', 'AveRooms': 'knn', 'default': 'median'}
  }
  ```
- Applies each cleaning step in order: duplicates → outliers → missing
- Returns cleaned DataFrame and a detailed log of changes made (how many rows removed, values imputed, etc.)

**Part D: End-to-End Test**

1. Apply the pipeline to the Wine dataset (`load_wine()`)
2. Introduce various quality issues (missing, duplicates, outliers)
3. Run the quality report and visualization
4. Clean the data with different configurations
5. Compare model performance (use any classifier) on raw vs. cleaned data
6. Write a brief summary (5-7 sentences) explaining:
   - What quality issues were found
   - Which cleaning strategies were chosen and why
   - How much model performance improved
   - What trade-offs were made (e.g., deleting rows vs. computational cost of KNN)

## Solutions

**Solution 1**

In [ ]:
from sklearn.datasets import load_breast_cancer
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set random seed
np.random.seed(42)

# Load dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)

print("Original shape:", df.shape)
print("\nOriginal data (first 5 rows):")
print(df.head())

# Introduce quality issues
# 1. Set 10% missing in two random features
features_to_corrupt = np.random.choice(df.columns, 2, replace=False)
print(f"\nIntroducing missing values in: {features_to_corrupt}")

for feature in features_to_corrupt:
    missing_mask = np.random.random(len(df)) < 0.10
    df.loc[missing_mask, feature] = np.nan

# 2. Duplicate 15 random rows
duplicate_indices = np.random.choice(df.index, 15, replace=False)
df = pd.concat([df, df.loc[duplicate_indices]], ignore_index=True)

# Detect and report quality issues
print("\n=== QUALITY REPORT ===")

# Missing values
print("\nMissing Values:")
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100
missing_report = pd.DataFrame({
    'Missing_Count': missing_counts,
    'Missing_Percent': missing_pct
})
print(missing_report[missing_report['Missing_Count'] > 0])

# Duplicates
dup_count = df.duplicated().sum()
print(f"\nDuplicate rows: {dup_count} ({dup_count/len(df)*100:.2f}%)")

# Basic statistics
print("\nBasic Statistics:")
print(df.describe())

# Visualize missing values
missing_viz = missing_report[missing_report['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

plt.figure(figsize=(10, 6))
plt.bar(range(len(missing_viz)), missing_viz['Missing_Count'], color='coral')
plt.xticks(range(len(missing_viz)), missing_viz.index, rotation=45, ha='right')
plt.xlabel('Feature')
plt.ylabel('Missing Count')
plt.title('Missing Values by Column')
plt.tight_layout()
plt.show()

This solution loads the Breast Cancer dataset, introduces realistic quality issues, and systematically detects them. The report shows missing value counts and percentages, identifies duplicates, and provides basic statistics. The bar chart visualizes missing values by column for easy identification of problematic features.

**Solution 2**

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

# Load dataset
data = load_diabetes()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

print("Original shape:", df.shape)

# Select 3 random features and introduce 20% missing values
features_to_corrupt = np.random.choice(data.feature_names, 3, replace=False)
print(f"Introducing 20% MCAR in: {features_to_corrupt}")

df_missing = df.copy()
for feature in features_to_corrupt:
    missing_mask = np.random.random(len(df)) < 0.20
    df_missing.loc[missing_mask, feature] = np.nan

# Split BEFORE imputation
X = df_missing.drop('target', axis=1)
y = df_missing['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTrain set: {X_train.shape}, Test set: {X_test.shape}")
print(f"Missing values in train: {X_train.isnull().sum().sum()}")
print(f"Missing values in test: {X_test.isnull().sum().sum()}")

# Compare strategies
results = {}

# Strategy 1: Deletion (listwise)
X_train_del = X_train.dropna()
y_train_del = y_train[X_train_del.index]
X_test_del = X_test.dropna()
y_test_del = y_test[X_test_del.index]

model = LinearRegression()
model.fit(X_train_del, y_train_del)
y_pred = model.predict(X_test_del)
rmse = np.sqrt(mean_squared_error(y_test_del, y_pred))
results['Deletion'] = rmse
print(f"\nDeletion: Kept {len(X_train_del)}/{len(X_train)} training samples, RMSE = {rmse:.4f}")

# Strategy 2: Mean Imputation
imputer_mean = SimpleImputer(strategy='mean')
X_train_mean = pd.DataFrame(imputer_mean.fit_transform(X_train), columns=X_train.columns)
X_test_mean = pd.DataFrame(imputer_mean.transform(X_test), columns=X_test.columns)

model = LinearRegression()
model.fit(X_train_mean, y_train)
y_pred = model.predict(X_test_mean)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
results['Mean Imputation'] = rmse
print(f"Mean Imputation: RMSE = {rmse:.4f}")

# Strategy 3: Median Imputation
imputer_median = SimpleImputer(strategy='median')
X_train_median = pd.DataFrame(imputer_median.fit_transform(X_train), columns=X_train.columns)
X_test_median = pd.DataFrame(imputer_median.transform(X_test), columns=X_test.columns)

model = LinearRegression()
model.fit(X_train_median, y_train)
y_pred = model.predict(X_test_median)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
results['Median Imputation'] = rmse
print(f"Median Imputation: RMSE = {rmse:.4f}")

# Strategy 4: KNN Imputation
imputer_knn = KNNImputer(n_neighbors=5)
X_train_knn = pd.DataFrame(imputer_knn.fit_transform(X_train), columns=X_train.columns)
X_test_knn = pd.DataFrame(imputer_knn.transform(X_test), columns=X_test.columns)

model = LinearRegression()
model.fit(X_train_knn, y_train)
y_pred = model.predict(X_test_knn)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
results['KNN Imputation'] = rmse
print(f"KNN Imputation: RMSE = {rmse:.4f}")

# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution comparison for one feature
feature_to_plot = features_to_corrupt[0]
axes[0].hist(df[feature_to_plot], bins=30, alpha=0.5, label='Original', color='blue')
axes[0].hist(X_train_mean[feature_to_plot], bins=30, alpha=0.5, label='Mean Imputed', color='green')
axes[0].hist(X_train_knn[feature_to_plot], bins=30, alpha=0.5, label='KNN Imputed', color='orange')
axes[0].set_xlabel(feature_to_plot)
axes[0].set_ylabel('Frequency')
axes[0].set_title(f'Distribution Comparison: {feature_to_plot}')
axes[0].legend()

# RMSE comparison
strategies_list = list(results.keys())
rmse_values = list(results.values())
best_strategy = min(results, key=results.get)
colors = ['coral' if s != best_strategy else 'green' for s in strategies_list]

axes[1].bar(strategies_list, rmse_values, color=colors)
axes[1].set_ylabel('RMSE (lower is better)')
axes[1].set_title('Comparison of Imputation Strategies')
axes[1].tick_params(axis='x', rotation=45)
for i, v in enumerate(rmse_values):
    axes[1].text(i, v + 0.5, f'{v:.2f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print(f"\nBest strategy: {best_strategy} (RMSE = {results[best_strategy]:.4f})")
print(f"\nTrade-offs: Deletion lost {len(X_train) - len(X_train_del)} training samples.")
print(f"KNN imputation is computationally expensive but preserved all samples and performed best.")

This solution demonstrates the critical importance of train-test split timing and compares four imputation strategies. KNN typically performs best because it uses feature relationships, but deletion trades sample size for simplicity. The visualizations show how different methods affect distributions and model performance. The trade-off analysis highlights that deletion loses valuable data while sophisticated methods like KNN add computational cost.

**Solution 3**

In [ ]:
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.impute import SimpleImputer, KNNImputer
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

# Part A: Quality Report Function
def generate_quality_report(df):
    """Generate comprehensive data quality report."""
    report = {}

    # Shape and memory
    report['shape'] = df.shape
    report['memory_usage_mb'] = df.memory_usage(deep=True).sum() / 1e6

    # Missing values
    missing = df.isnull().sum()
    report['missing_values'] = {
        'counts': missing.to_dict(),
        'percentages': (missing / len(df) * 100).to_dict()
    }

    # Duplicates
    report['duplicates'] = df.duplicated().sum()

    # Outliers
    report['outliers'] = {}
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    for col in numeric_cols:
        # IQR method
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers_iqr = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()

        # Z-score method
        z_scores = np.abs((df[col] - df[col].mean()) / df[col].std())
        outliers_z = (z_scores > 3).sum()

        report['outliers'][col] = {'iqr': outliers_iqr, 'zscore': outliers_z}

    # Data types
    report['dtypes'] = df.dtypes.to_dict()

    # Suspicious patterns
    report['suspicious'] = {}
    for col in numeric_cols:
        # All zeros
        all_zeros = (df[col] == 0).sum()
        if all_zeros / len(df) > 0.5:
            report['suspicious'][f'{col}_all_zeros'] = all_zeros

        # All same value
        if df[col].nunique() == 1:
            report['suspicious'][f'{col}_constant'] = True

        # Negative where impossible (example: age, price)
        if (df[col] < 0).any():
            report['suspicious'][f'{col}_negative'] = (df[col] < 0).sum()

    return report

# Part B: Visualization Dashboard
def visualize_quality(df, report):
    """Generate quality visualization dashboard."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    # 1. Missing data bar chart
    missing_df = pd.DataFrame({
        'Feature': list(report['missing_values']['counts'].keys()),
        'Count': list(report['missing_values']['counts'].values())
    })
    missing_df = missing_df[missing_df['Count'] > 0].sort_values('Count', ascending=False)

    if len(missing_df) > 0:
        axes[0, 0].bar(range(len(missing_df)), missing_df['Count'], color='coral')
        axes[0, 0].set_xticks(range(len(missing_df)))
        axes[0, 0].set_xticklabels(missing_df['Feature'], rotation=45, ha='right')
        axes[0, 0].set_title('Missing Values by Column')
        axes[0, 0].set_ylabel('Count')
    else:
        axes[0, 0].text(0.5, 0.5, 'No Missing Values', ha='center', va='center')
        axes[0, 0].set_title('Missing Values by Column')

    # 2. Missing data heatmap
    if df.isnull().sum().sum() > 0:
        sns.heatmap(df.head(100).isnull(), cbar=False, cmap='viridis', ax=axes[0, 1])
        axes[0, 1].set_title('Missing Data Pattern (First 100 Rows)')
    else:
        axes[0, 1].text(0.5, 0.5, 'No Missing Values', ha='center', va='center')
        axes[0, 1].set_title('Missing Data Pattern')

    # 3. Box plots for numeric features
    numeric_cols = df.select_dtypes(include=[np.number]).columns[:5]  # Limit to 5
    if len(numeric_cols) > 0:
        df[numeric_cols].boxplot(ax=axes[0, 2])
        axes[0, 2].set_title('Outliers in Numeric Features')
        axes[0, 2].tick_params(axis='x', rotation=45)

    # 4. Correlation heatmap of missingness
    missing_corr = df.isnull().corr()
    if missing_corr.sum().sum() > 0:
        sns.heatmap(missing_corr, cmap='coolwarm', center=0, ax=axes[1, 0],
                    square=True, linewidths=1)
        axes[1, 0].set_title('Missingness Correlation')
    else:
        axes[1, 0].text(0.5, 0.5, 'No Missing Values', ha='center', va='center')
        axes[1, 0].set_title('Missingness Correlation')

    # 5. Distribution histograms for features with outliers
    outlier_features = [k for k, v in report['outliers'].items() if v['iqr'] > 0][:3]
    if len(outlier_features) > 0:
        for i, col in enumerate(outlier_features):
            axes[1, 1].hist(df[col].dropna(), bins=30, alpha=0.6, label=col)
        axes[1, 1].set_title('Distribution of Features with Outliers')
        axes[1, 1].legend()
    else:
        axes[1, 1].text(0.5, 0.5, 'No Outliers Detected', ha='center', va='center')
        axes[1, 1].set_title('Distribution of Features with Outliers')

    # 6. Quality score summary
    total_missing_pct = sum(report['missing_values']['percentages'].values()) / len(df.columns)
    duplicate_pct = (report['duplicates'] / len(df)) * 100
    total_outliers = sum([v['iqr'] for v in report['outliers'].values()])
    outlier_pct = (total_outliers / (len(df) * len(numeric_cols))) * 100 if len(numeric_cols) > 0 else 0

    quality_score = max(0, 100 - total_missing_pct - duplicate_pct - outlier_pct)

    axes[1, 2].bar(['Quality Score'], [quality_score], color='green' if quality_score > 70 else 'orange')
    axes[1, 2].set_ylim(0, 100)
    axes[1, 2].set_title('Overall Data Quality Score')
    axes[1, 2].set_ylabel('Score (0-100)')
    axes[1, 2].axhline(70, color='red', linestyle='--', label='Threshold')
    axes[1, 2].text(0, quality_score + 3, f'{quality_score:.1f}', ha='center')
    axes[1, 2].legend()

    plt.tight_layout()
    plt.show()

# Part C: Automated Cleaning Function
def clean_data(df, config):
    """Clean data according to configuration."""
    df_clean = df.copy()
    log = []

    # 1. Handle duplicates
    if config.get('duplicates') == 'remove':
        before = len(df_clean)
        df_clean = df_clean.drop_duplicates()
        after = len(df_clean)
        log.append(f"Removed {before - after} duplicate rows")

    # 2. Handle outliers
    outlier_config = config.get('outliers', {})
    if outlier_config.get('method') == 'winsorize':
        percentiles = outlier_config.get('percentiles', (0.01, 0.99))
        numeric_cols = df_clean.select_dtypes(include=[np.number]).columns

        for col in numeric_cols:
            before_min = df_clean[col].min()
            before_max = df_clean[col].max()

            p_low = df_clean[col].quantile(percentiles[0])
            p_high = df_clean[col].quantile(percentiles[1])
            df_clean[col] = df_clean[col].clip(lower=p_low, upper=p_high)

            if before_min != df_clean[col].min() or before_max != df_clean[col].max():
                log.append(f"Winsorized '{col}' to [{p_low:.2f}, {p_high:.2f}]")

    elif outlier_config.get('method') == 'remove':
        numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
        before = len(df_clean)

        for col in numeric_cols:
            Q1 = df_clean[col].quantile(0.25)
            Q3 = df_clean[col].quantile(0.75)
            IQR = Q3 - Q1
            df_clean = df_clean[
                (df_clean[col] >= Q1 - 1.5 * IQR) &
                (df_clean[col] <= Q3 + 1.5 * IQR)
            ]

        after = len(df_clean)
        if before != after:
            log.append(f"Removed {before - after} rows with outliers")

    # 3. Handle missing values (must be done last, after split in real use)
    missing_config = config.get('missing', {})
    default_strategy = missing_config.get('default', 'median')

    for col in df_clean.columns:
        if df_clean[col].isnull().sum() > 0:
            strategy = missing_config.get(col, default_strategy)

            if strategy == 'mean':
                fill_value = df_clean[col].mean()
                df_clean[col].fillna(fill_value, inplace=True)
                log.append(f"Imputed '{col}' with mean ({fill_value:.2f})")

            elif strategy == 'median':
                fill_value = df_clean[col].median()
                df_clean[col].fillna(fill_value, inplace=True)
                log.append(f"Imputed '{col}' with median ({fill_value:.2f})")

            elif strategy == 'knn':
                # Note: In real use, fit on train only
                imputer = KNNImputer(n_neighbors=5)
                df_clean[[col]] = imputer.fit_transform(df_clean[[col]])
                log.append(f"Imputed '{col}' with KNN")

    return df_clean, log

# Part D: End-to-End Test
print("=== LOADING WINE DATASET ===")
data = load_wine()
df_wine = pd.DataFrame(data.data, columns=data.feature_names)
df_wine['target'] = data.target

print(f"Original shape: {df_wine.shape}")

# Introduce quality issues
print("\n=== INTRODUCING QUALITY ISSUES ===")
df_corrupted = df_wine.copy()

# Missing values
for col in np.random.choice(df_wine.columns[:-1], 3, replace=False):
    mask = np.random.random(len(df_corrupted)) < 0.15
    df_corrupted.loc[mask, col] = np.nan
    print(f"Introduced 15% missing in '{col}'")

# Duplicates
dup_indices = np.random.choice(df_corrupted.index, 20, replace=False)
df_corrupted = pd.concat([df_corrupted, df_corrupted.loc[dup_indices]], ignore_index=True)
print(f"Added 20 duplicate rows")

# Outliers
for col in np.random.choice(df_wine.columns[:-1], 2, replace=False):
    outlier_indices = np.random.choice(df_corrupted.index, 10, replace=False)
    df_corrupted.loc[outlier_indices, col] *= 3
    print(f"Created outliers in '{col}'")

# Generate quality report
print("\n=== QUALITY REPORT ===")
report = generate_quality_report(df_corrupted)
print(f"Shape: {report['shape']}")
print(f"Duplicates: {report['duplicates']}")
print(f"Total missing values: {sum(report['missing_values']['counts'].values())}")

# Visualize
visualize_quality(df_corrupted, report)

# Clean data
print("\n=== CLEANING DATA ===")
config = {
    'duplicates': 'remove',
    'outliers': {'method': 'winsorize', 'percentiles': (0.01, 0.99)},
    'missing': {'default': 'knn'}
}

df_cleaned, cleaning_log = clean_data(df_corrupted, config)

print("\nCleaning Log:")
for entry in cleaning_log:
    print(f"  - {entry}")

# Compare model performance
print("\n=== COMPARING MODEL PERFORMANCE ===")

# Raw data (drop missing rows for fair comparison)
X_raw = df_corrupted.drop('target', axis=1).dropna()
y_raw = df_corrupted.loc[X_raw.index, 'target']
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

model_raw = RandomForestClassifier(random_state=42)
model_raw.fit(X_train_raw, y_train_raw)
acc_raw = accuracy_score(y_test_raw, model_raw.predict(X_test_raw))
print(f"Raw data accuracy: {acc_raw:.4f} (using {len(X_train_raw)} training samples)")

# Cleaned data
X_clean = df_cleaned.drop('target', axis=1)
y_clean = df_cleaned['target']
X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=42
)

model_clean = RandomForestClassifier(random_state=42)
model_clean.fit(X_train_clean, y_train_clean)
acc_clean = accuracy_score(y_test_clean, model_clean.predict(X_test_clean))
print(f"Cleaned data accuracy: {acc_clean:.4f} (using {len(X_train_clean)} training samples)")

improvement = (acc_clean - acc_raw) / acc_raw * 100
print(f"\nPerformance improvement: {improvement:.2f}%")

# Summary
print("\n=== SUMMARY ===")
summary = f"""
The Wine dataset analysis revealed multiple quality issues: {report['duplicates']} duplicate
rows ({report['duplicates']/len(df_corrupted)*100:.1f}%), {sum(report['missing_values']['counts'].values())}
missing values across multiple features, and outliers in several numeric columns.

The cleaning strategy employed duplicate removal, winsorization for outliers (capping at 1st and
99th percentiles), and KNN imputation for missing values. KNN was chosen over simpler methods
because it preserves feature relationships, which is important for multivariate datasets like Wine.

Model performance improved from {acc_raw:.4f} to {acc_clean:.4f} accuracy ({improvement:.2f}% improvement).
The raw data approach lost {len(df_corrupted.dropna())} samples due to listwise deletion, while
the cleaned approach preserved all samples after duplicate removal.

The main trade-off was computational cost: KNN imputation is slower than mean/median, but the
performance gain justified the extra computation time. Winsorization was preferred over deletion
to preserve sample size, accepting slight distribution distortion as a reasonable trade-off.
"""
print(summary)

This comprehensive solution provides a complete data quality pipeline with three reusable functions and end-to-end testing. The quality report captures all key metrics, the visualization dashboard provides immediate visual insight into data issues, and the cleaning function is configurable for different scenarios. The Wine dataset test demonstrates how systematic cleaning improves model performance while highlighting trade-offs between computational cost and accuracy.

## Key Takeaways

- **Data quality determines model quality.** Even the best algorithm cannot extract reliable insights from fundamentally flawed data. Companies lose 15-25% of revenue to poor data quality, and most ML projects fail due to data issues, not algorithmic ones.

- **Understanding *why* data is missing guides *how* to fix it.** MCAR (truly random) can use simple imputation or deletion. MAR (related to observed data) benefits from sophisticated imputation like KNN or Iterative methods. MNAR (related to the unobserved value) requires special handling, often by flagging missingness as a separate feature.

- **Split data before any preprocessing.** This is critical to avoid data leakage. Calculate all statistics (means for imputation, scaling parameters, etc.) on the training set only, then apply them to both train and test sets. Use scikit-learn Pipelines to enforce this automatically.

- **Not all outliers are errors; not all errors are outliers.** Investigate before deciding. Domain knowledge is essential. When uncertain, use robust methods like winsorization (capping at percentiles) that reduce influence without deletion, or use algorithms naturally resistant to outliers (tree-based models).

- **Data cleaning is iterative, not linear.** Issues are discovered during EDA, cleaned, and then more subtle issues found. Visualize extensively—patterns invisible in tables become obvious in plots. Validate that cleaning actually improved downstream performance; don't assume it helped.

- **Document every decision.** Future reviewers need to understand what was done and why. Write scripts, not manual Excel edits. Make pipelines reproducible, configurable, and maintainable. Log what was changed, how many values were affected, and the rationale behind each choice.

**Next:** Chapter 12 covers feature engineering techniques for creating informative predictive variables from raw data.